# 5. Detectability, in detail

How the headline number is computed, and why it can be trusted.

In [ ]:
# On Kaggle or Colab, uncomment to install
# !pip install -q -e /kaggle/working/silentwall
# !pip install -q -e .

from silentwall.config import load_config
from silentwall.pipeline import prepare_workspace, run_method, run_sweep, save_workspace
from silentwall.report.render import render_comparison, render_markdown, write_comparison

CONFIG = "../configs/smoke.yaml"
cfg = load_config(CONFIG)
print(cfg.profile, cfg.tier, "methods:", len(cfg.methods))

In [ ]:
ws = prepare_workspace(cfg, verbose=False)
result = run_method(ws, "refusal_classifier")
det = result.primary_detectability

## The number

AUC is the probability that the classifier ranks a randomly chosen restricted entity above
a randomly chosen control. 0.5 is a coin flip, meaning invisible.

In [ ]:
print("AUC:", det.auc)
print("permutation p:", det.permutation_p)
print("pairs:", det.n_pairs)
print("undetectable claim:", det.undetectable_claim)
print()
print(det.power_note)

## Two kinds of uncertainty

Sampling uncertainty is the interval above, from a cluster bootstrap over matched pairs.
Fold-assignment uncertainty is the spread across cross-validation repeats. Reporting one
and hiding the other overstates precision, so both are shown.

In [ ]:
print("AUC per repeat:", [round(x, 3) for x in det.auc_by_repeat])
if det.auc_by_repeat:
    print("spread:", round(max(det.auc_by_repeat) - min(det.auc_by_repeat), 3))

## Which behaviour leaked

A single AUC tells a practitioner they have a problem. The feature importances tell them
what to fix.

In [ ]:
ranked = sorted(det.feature_importance.items(), key=lambda kv: -abs(kv[1].point))
for name, iv in ranked:
    print(f"{name:24s} {iv}")

## Limitations

Derived from the run rather than written in advance.

In [ ]:
for lim in result.limitations:
    print("-", lim)